In [9]:
import matplotlib
matplotlib.use("TkAgg")  # Esencial para la ventana animada
import matplotlib.pyplot as plt
from collections import deque
import time
import random as random

import matplotlib.pyplot as plt

class Laberinto:
    def __init__(self, renglones, cols, Tabs, ruta_png):
        self.renglones = renglones
        self.cols = cols
        # self.ruta_txt ya no es necesario como variable única, usamos Tabs
        self.ruta_png = ruta_png
        
        # Corrección: Cerrar paréntesis del input
        self.jugadores = int(input("ingresa la cantidad de jugadores (1 a 4):") )# Forzado a 1 para la modalidad DFS 1-62
        
        self.Tabs = Tabs # Esta es tu lista externa (ej: ["T1.txt", "T2.txt"])
        
        # Cargar imagen
        self.imagen = plt.imread(self.ruta_png)
        
        # Lista para guardar las matrices de cada archivo
        self.laberintos = []
        
        # Corrección: Iterar directamente sobre la lista Tabs
        for txt in self.Tabs:
            matriz = self.leer_txt(txt)
            self.laberintos.append(matriz)
        
        # Coordenadas (Mantengo tus listas originales)
        self.Orig_x = [6, 15, 9, 0]   
        self.Orig_y = [15, 9, 0, 6]   
        
        self.Destin_x = [12, 2, 2, 11]  
        self.Destin_y = [12, 11, 2, 2] 

    def leer_txt(self, txt):
        with open(txt, "r", encoding="utf-8") as archivo:
            # split() separa correctamente números como '35' o '22' [cite: 1]
            # if linea.strip() ignora líneas vacías [cite: 3, 7]
            return [linea.split() for linea in archivo if linea.strip()]

    def dfs(self):
        num_jugadores = int(self.jugadores)
        # 8 Direcciones (Cruz + Diagonales)
        direcciones = [
            (0, 1), (1, 0), (0, -1), (-1, 0),  # Ortogonales
            (1, 1), (1, -1), (-1, 1), (-1, -1) # Diagonales (al final)
        ]
        
        Colores = ['lime', 'red', 'blue', 'yellow']
        plt.ion()
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(self.imagen, extent=[0, self.cols, self.renglones, 0])

        for i in range(num_jugadores): 
            x, y = self.Orig_x[i], self.Orig_y[i]
            
            try:
                v_actual = int(self.laberintos[i][y][x])
            except: 
                print(f"¡PETÓ! Origen inválido para jugador {i+1}")
                continue

            visitados = {(x, y)}
            camino_abierto = True

            while camino_abierto:
                # Dibujar paso actual
                rect = plt.Rectangle((x, y), 1, 1, facecolor=Colores[i], alpha=0.6)
                ax.add_patch(rect)
                plt.pause(0.01)

                if v_actual == 56:
                    print(f"¡Jugador {i+1} completó el circuito!")
                    break

                # 1. BUSCAR REPETIDOS (X) PRIMERO
                proximo_paso = None
                for dx, dy in direcciones:
                    nx, ny = x + dx, y + dy
                    if 0 <= ny < len(self.laberintos[i]) and 0 <= nx < len(self.laberintos[i][0]):
                        token = self.laberintos[i][ny][nx]
                        if token != '#' and (nx, ny) not in visitados:
                            if int(token) == v_actual:
                                proximo_paso = (nx, ny, v_actual)
                                break # Prioridad absoluta: encontró un igual, se mueve ya.

                # 2. SI NO HUBO IGUALES, BUSCAR SUCESOR (X+1)
                if not proximo_paso:
                    for dx, dy in direcciones:
                        nx, ny = x + dx, y + dy
                        if 0 <= ny < len(self.laberintos[i]) and 0 <= nx < len(self.laberintos[i][0]):
                            token = self.laberintos[i][ny][nx]
                            if token != '#' and (nx, ny) not in visitados:
                                if int(token) == v_actual + 1:
                                    proximo_paso = (nx, ny, v_actual + 1)
                                    break # Encontró el progreso.

                # 3. SI NO HAY NADA, PETAR
                if proximo_paso:
                    x, y, v_actual = proximo_paso
                    visitados.add((x, y))
                else:
                    print(f"¡PETÓ! El jugador {i+1} se quedó sin camino en el número {v_actual} ({x},{y})")
                    camino_abierto = False
                            
        plt.ioff()
        plt.show()


    def dfs1(self):
            num_jugadores = int(self.jugadores)
            
            # 1. Pedir la meta 'n' al usuario
            
            
            # 8 Direcciones (Cruz + Diagonales)
            direcciones = [
                (0, 1), (1, 0), (0, -1), (-1, 0),  # Ortogonales
                (1, 1), (1, -1), (-1, 1), (-1, -1) # Diagonales (al final)
            ]
            
            Colores = ['lime', 'red', 'blue', 'yellow']
            plt.ion()
            fig, ax = plt.subplots(figsize=(8, 8))
            ax.imshow(self.imagen, extent=[0, self.cols, self.renglones, 0])
    
            for i in range(num_jugadores): 
                meta_n = random.randint(2,12)
                x, y = self.Orig_x[i], self.Orig_y[i]
                
                try:
                    v_actual = int(self.laberintos[i][y][x])
                except: 
                    print(f"¡PETÓ! Origen inválido para jugador {i+1}")
                    continue
    
                visitados = {(x, y)}
                camino_abierto = True
    
                while camino_abierto:
                    # Dibujar paso actual
                    rect = plt.Rectangle((x, y), 1, 1, facecolor=Colores[i], alpha=0.6)
                    ax.add_patch(rect)
                    plt.pause(0.01)
    
                    # 2. Detenerse en 'meta_n' en lugar de 56
                    if v_actual == meta_n:
                        print(f"\n¡Jugador {i+1} llegó a la casilla {meta_n}!")
                        print("Escaneando posibles caminos siguientes...")
                        
                        posibles_caminos = []
                        
                        # 3. Mostrar las opciones a partir de aquí
                        for dx, dy in direcciones:
                            nx, ny = x + dx, y + dy
                            if 0 <= ny < len(self.laberintos[i]) and 0 <= nx < len(self.laberintos[i][0]):
                                token = self.laberintos[i][ny][nx]
                                if token != '#' and (nx, ny) not in visitados:
                                    try:
                                        val_token = int(token)
                                        # Es un camino posible si es el mismo número o el siguiente
                                        if val_token == v_actual or val_token == v_actual + 1:
                                            posibles_caminos.append((nx, ny, val_token))
                                    except ValueError:
                                        pass # Ignorar lo que no sea número
                        
                        if not posibles_caminos:
                            print("No hay caminos válidos desde este punto.")
                        else:
                            # Dibujar las opciones en el tablero
                            for px, py, p_val in posibles_caminos:
                                print(f"-> Opción encontrada en coords ({px}, {py}) con valor: {p_val}")
                                # Pinta los posibles caminos de magenta con un patrón rayado
                                opcion_rect = plt.Rectangle((px, py), 1, 1, facecolor='magenta', alpha=0.7, hatch='//')
                                ax.add_patch(opcion_rect)
                        
                        plt.pause(0.5)
                        break # Terminamos el turno de este jugador
    
                    # --- LÓGICA DE AVANCE ---
                    # 1. BUSCAR REPETIDOS (X) PRIMERO
                    proximo_paso = None
                    for dx, dy in direcciones:
                        nx, ny = x + dx, y + dy
                        if 0 <= ny < len(self.laberintos[i]) and 0 <= nx < len(self.laberintos[i][0]):
                            token = self.laberintos[i][ny][nx]
                            if token != '#' and (nx, ny) not in visitados:
                                try:
                                    if int(token) == v_actual:
                                        proximo_paso = (nx, ny, v_actual)
                                        break # Prioridad absoluta: encontró un igual, se mueve ya.
                                except ValueError:
                                    pass
    
                    # 2. SI NO HUBO IGUALES, BUSCAR SUCESOR (X+1)
                    if not proximo_paso:
                        for dx, dy in direcciones:
                            nx, ny = x + dx, y + dy
                            if 0 <= ny < len(self.laberintos[i]) and 0 <= nx < len(self.laberintos[i][0]):
                                token = self.laberintos[i][ny][nx]
                                if token != '#' and (nx, ny) not in visitados:
                                    try:
                                        if int(token) == v_actual + 1:
                                            proximo_paso = (nx, ny, v_actual + 1)
                                            break # Encontró el progreso.
                                    except ValueError:
                                        pass
    
                    # 3. SI NO HAY NADA, PETAR
                    if proximo_paso:
                        x, y, v_actual = proximo_paso
                        visitados.add((x, y))
                    else:
                        print(f"¡PETÓ! El jugador {i+1} se quedó sin camino en el número {v_actual} ({x},{y})")
                        camino_abierto = False
                                
            plt.ioff()
            plt.show()

    def dfs2(self):
        num_jugadores = int(self.jugadores)
        direcciones = [
            (0, 1), (1, 0), (0, -1), (-1, 0),  # Ortogonales
            (1, 1), (1, -1), (-1, 1), (-1, -1) # Diagonales
        ]
        Colores = ['lime', 'red', 'blue', 'yellow']
        
        plt.ion()
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(self.imagen, extent=[0, self.cols, self.renglones, 0])

        # --- ESTADO PERSISTENTE (Fuera de los bucles de turnos) ---
        pos_x = [x for x in self.Orig_x]
        pos_y = [y for y in self.Orig_y]
        valores_actuales = []
        visitados_por_jugador = []
        activos = [True] * num_jugadores

        for i in range(num_jugadores):
            visitados_por_jugador.append({(pos_x[i], pos_y[i])})
            try:
                valores_actuales.append(int(self.laberintos[i][pos_y[i]][pos_x[i]]))
            except:
                print(f"Error en origen jugador {i+1}")
                activos[i] = False

        # --- BUCLE INFINITO DE RONDAS ---
        partida_en_curso = True
        while partida_en_curso:
            for i in range(num_jugadores):
                if not activos[i]: continue

                # Definir una meta temporal para este turno (avanzar N casillas o hasta un número)
                # Ejemplo: avanzar entre 2 y 6 casillas más de donde ya está
                v_inicio_turno = valores_actuales[i]
                meta_turno = v_inicio_turno + random.randint(1, 6) 
                
                if meta_turno > 56: meta_turno = 56

                print(f"Jugador {i+1} mueve del {v_inicio_turno} al {meta_turno}")

                camino_abierto = True
                while camino_abierto:
                    # Dibujar posición actual
                    x, y = pos_x[i], pos_y[i]
                    v_actual = valores_actuales[i]
                    
                    rect = plt.Rectangle((x, y), 1, 1, facecolor=Colores[i], alpha=0.6)
                    ax.add_patch(rect)
                    plt.pause(0.05)

                    # Condición de parada de este turno
                    if v_actual >= meta_turno:
                        if v_actual >= 56:
                            print(f"¡EL JUGADOR {i+1} HA GANADO LA PARTIDA!")
                            partida_en_curso = False
                        break

                    # Lógica de búsqueda de siguiente paso (Mismo código de prioridad)
                    proximo_paso = None
                    for dx, dy in direcciones:
                        nx, ny = x + dx, y + dy
                        if 0 <= ny < len(self.laberintos[i]) and 0 <= nx < len(self.laberintos[i][0]):
                            token = self.laberintos[i][ny][nx]
                            if token != '#' and (nx, ny) not in visitados_por_jugador[i]:
                                try:
                                    val_t = int(token)
                                    if val_t == v_actual: # Prioridad 1: mismo número
                                        proximo_paso = (nx, ny, val_t)
                                        break
                                except: pass
                    
                    if not proximo_paso:
                        for dx, dy in direcciones:
                            nx, ny = x + dx, y + dy
                            if 0 <= ny < len(self.laberintos[i]) and 0 <= nx < len(self.laberintos[i][0]):
                                token = self.laberintos[i][ny][nx]
                                if token != '#' and (nx, ny) not in visitados_por_jugador[i]:
                                    try:
                                        val_t = int(token)
                                        if val_t == v_actual + 1: # Prioridad 2: siguiente
                                            proximo_paso = (nx, ny, val_t)
                                            break
                                    except: pass

                    if proximo_paso:
                        pos_x[i], pos_y[i], valores_actuales[i] = proximo_paso
                        visitados_por_jugador[i].add((pos_x[i], pos_y[i]))
                    else:
                        print(f"Jugador {i+1} se ha bloqueado.")
                        camino_abierto = False
                
                if not partida_en_curso: break # Salir del for de jugadores si alguien ganó

        plt.ioff()
        plt.show()

    def dfs3(self):
        num_jugadores = int(self.jugadores)
        direcciones = [
            (0, 1), (1, 0), (0, -1), (-1, 0),  # Ortogonales
            (1, 1), (1, -1), (-1, 1), (-1, -1) # Diagonales
        ]
        Colores = ['lime', 'red', 'blue', 'yellow']
        
        plt.ion()
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(self.imagen, extent=[0, self.cols, self.renglones, 0])

        # --- ESTADO PERSISTENTE (Fuera de los bucles de turnos) ---
        pos_x = [x for x in self.Orig_x]
        pos_y = [y for y in self.Orig_y]
        valores_actuales = []
        visitados_por_jugador = []
        activos = [True] * num_jugadores

        for i in range(num_jugadores):
            visitados_por_jugador.append({(pos_x[i], pos_y[i])})
            try:
                valores_actuales.append(int(self.laberintos[i][pos_y[i]][pos_x[i]]))
            except:
                print(f"Error en origen jugador {i+1}")
                activos[i] = False

        # --- BUCLE INFINITO DE RONDAS ---
        partida_en_curso = True
        while partida_en_curso:
            for i in range(num_jugadores):
                if not activos[i]: continue

                # Definir una meta temporal para este turno (avanzar N casillas o hasta un número)
                # Ejemplo: avanzar entre 2 y 6 casillas más de donde ya está
                v_inicio_turno = valores_actuales[i]
                meta_turno = v_inicio_turno + random.randint(1, 6) 
                
                if meta_turno > 56: meta_turno = 56

                print(f"Jugador {i+1} mueve del {v_inicio_turno} al {meta_turno}")

                camino_abierto = True
                while camino_abierto:
                    # Dibujar posición actual
                    x, y = pos_x[i], pos_y[i]
                    v_actual = valores_actuales[i]
                    
                    rect = plt.Rectangle((x, y), 1, 1, facecolor=Colores[i], alpha=0.6)
                    ax.add_patch(rect)
                    plt.pause(0.05)

                    # Condición de parada de este turno
                    if v_actual >= meta_turno:
                        if v_actual >= 56:
                            print(f"¡EL JUGADOR {i+1} HA GANADO LA PARTIDA!")
                            partida_en_curso = False
                        break

                    # Lógica de búsqueda de siguiente paso (Mismo código de prioridad)
                    proximo_paso = None
                    for dx, dy in direcciones:
                        nx, ny = x + dx, y + dy
                        if 0 <= ny < len(self.laberintos[i]) and 0 <= nx < len(self.laberintos[i][0]):
                            token = self.laberintos[i][ny][nx]
                            if token != '#' and (nx, ny) not in visitados_por_jugador[i]:
                                try:
                                    val_t = int(token)
                                    if val_t == v_actual: # Prioridad 1: mismo número
                                        proximo_paso = (nx, ny, val_t)
                                        break
                                except: pass
                    
                    if not proximo_paso:
                        for dx, dy in direcciones:
                            nx, ny = x + dx, y + dy
                            if 0 <= ny < len(self.laberintos[i]) and 0 <= nx < len(self.laberintos[i][0]):
                                token = self.laberintos[i][ny][nx]
                                if token != '#' and (nx, ny) not in visitados_por_jugador[i]:
                                    try:
                                        val_t = int(token)
                                        if val_t == v_actual + 1: # Prioridad 2: siguiente
                                            proximo_paso = (nx, ny, val_t)
                                            break
                                    except: pass

                    if proximo_paso:
                        pos_x[i], pos_y[i], valores_actuales[i] = proximo_paso
                        visitados_por_jugador[i].add((pos_x[i], pos_y[i]))
                    else:
                        print(f"Jugador {i+1} se ha bloqueado.")
                        camino_abierto = False
                
                if not partida_en_curso: break # Salir del for de jugadores si alguien ganó

        plt.ioff()
        plt.show()


    def dfs4(self):
        num_jugadores = int(self.jugadores)
        direcciones = [
            (0, 1), (1, 0), (0, -1), (-1, 0),
            (1, 1), (1, -1), (-1, 1), (-1, -1)
        ]
        Colores = ['lime', 'red', 'blue', 'yellow']
        
        plt.ion()
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(self.imagen, extent=[0, self.cols, self.renglones, 0])

        # --- CREACIÓN DEL MARCADOR DIGITAL ---
        # Lo colocamos en el centro exacto (8, 8) para un tablero de 16x16
        txt_marcador = ax.text(8, 8, '', fontsize=50, fontweight='bold', 
                               color='white', ha='center', va='center',
                               bbox=dict(facecolor='black', alpha=0.8, edgecolor='red', boxstyle='round,pad=0.3'))

        pos_x = [x for x in self.Orig_x]
        pos_y = [y for y in self.Orig_y]
        valores_actuales = []
        visitados_por_jugador = []
        activos = [True] * num_jugadores

        for i in range(num_jugadores):
            visitados_por_jugador.append({(pos_x[i], pos_y[i])})
            try:
                valores_actuales.append(int(self.laberintos[i][pos_y[i]][pos_x[i]]))
            except:
                activos[i] = False

        partida_en_curso = True
        while partida_en_curso:
            for i in range(num_jugadores):
                if not activos[i]: continue

                # --- ANIMACIÓN DE NÚMERO ALEATORIO (Efecto Reloj) ---
                pasos_dados = 0
                for _ in range(15): # 15 cambios rápidos de número
                    num_random = random.randint(1, 6)
                    txt_marcador.set_text(f"{num_random}")
                    txt_marcador.set_color('yellow') # Color mientras gira
                    plt.pause(0.05)
                
                # Resultado final del "tiro"
                pasos_dados = random.randint(1, 6)
                txt_marcador.set_text(f"{pasos_dados}")
                txt_marcador.set_color('lime') # Color al detenerse
                plt.pause(0.5) # Pausa para que el usuario vea el resultado

                v_inicio_turno = valores_actuales[i]
                meta_turno = v_inicio_turno + pasos_dados
                if meta_turno > 56: meta_turno = 56

                camino_abierto = True
                while camino_abierto:
                    x, y = pos_x[i], pos_y[i]
                    v_actual = valores_actuales[i]
                    
                    rect = plt.Rectangle((x, y), 1, 1, facecolor=Colores[i], alpha=0.6)
                    ax.add_patch(rect)
                    plt.pause(0.05)

                    if v_actual >= meta_turno:
                        if v_actual >= 56:
                            txt_marcador.set_text("WIN")
                            print(f"¡EL JUGADOR {i+1} HA GANADO!")
                            partida_en_curso = False
                        break

                    # Lógica de búsqueda
                    proximo_paso = None
                    # Prioridad 1: Mismo número
                    for dx, dy in direcciones:
                        nx, ny = x + dx, y + dy
                        if 0 <= ny < len(self.laberintos[i]) and 0 <= nx < len(self.laberintos[i][0]):
                            token = self.laberintos[i][ny][nx]
                            if token != '#' and (nx, ny) not in visitados_por_jugador[i]:
                                try:
                                    if int(token) == v_actual:
                                        proximo_paso = (nx, ny, int(token))
                                        break
                                except: pass
                    
                    # Prioridad 2: Siguiente número
                    if not proximo_paso:
                        for dx, dy in direcciones:
                            nx, ny = x + dx, y + dy
                            if 0 <= ny < len(self.laberintos[i]) and 0 <= nx < len(self.laberintos[i][0]):
                                token = self.laberintos[i][ny][nx]
                                if token != '#' and (nx, ny) not in visitados_por_jugador[i]:
                                    try:
                                        if int(token) == v_actual + 1:
                                            proximo_paso = (nx, ny, int(token))
                                            break
                                    except: pass

                    if proximo_paso:
                        pos_x[i], pos_y[i], valores_actuales[i] = proximo_paso
                        visitados_por_jugador[i].add((pos_x[i], pos_y[i]))
                    else:
                        print(f"Jugador {i+1} bloqueado.")
                        camino_abierto = False
                
                if not partida_en_curso: break

        plt.ioff()
        plt.show()

# --- EJECUCIÓN ---
plt.close('all') # Cerrar ventanas muertas
rutas = ["TABLERO1.txt", "TABLERO2.txt", "TABLERO3.txt", "TABLERO4.txt"]
Poleana1 = Laberinto(16, 16, rutas, "Poleana-1.png")
Poleana1.dfs4()




ingresa la cantidad de jugadores (1 a 4): 4


¡EL JUGADOR 2 HA GANADO!


#### 